# UAB · Day 1: From Python to differential equations

**Student exercises.** Replace the TODOs, then run the check cell after each exercise.

Write a derivative, solve an initial-value problem, and verify the result against mathematics and conservation.

Deliver a verified two-state reaction model with a convergence study.

Time is in hours unless specified; model concentrations are g/L and volume is L.

Source: YeastFermentation Days 1–3. Python translation: Johan Le Nepvou De Carfort. Original MATLAB teaching material: Pedram Ramin and Krist Gernaey, PROSYS, DTU.

Run the embedded support-file setup once. For local Jupyter install numpy, scipy, and matplotlib first.


In [ ]:
# These support files are embedded for portable, offline notebook use.
# Original files retain their source content; course_models.py documents adapters.
import json
from pathlib import Path
_support = json.loads("{\"course_models.py\":\"\\\"\\\"\\\"Teaching adapters. Original equations/parameters remain in course_source.\\n\\nTime: h; concentration: g/L; volume: L; feed: L/h.\\nThe event/phase handling here corrects integration across feed discontinuities.\\n\\\"\\\"\\\"\\nimport random\\nimport numpy as np\\nfrom scipy.integrate import solve_ivp\\nfrom course_source.DAY3.parameters import Parameters as YeastParameters\\nfrom course_source.DAY3.model import initial_values, feed_values\\nfrom course_source.DAY3 import model as source_model\\n\\nSTATE_NAMES = ['Glucose', 'Xylose', 'Furfural', 'Furfuryl alcohol',\\n               '5-HMF', 'Acetic acid', 'Ethanol', 'Biomass', 'Volume']\\n\\ndef yeast_rhs(t, y, par, feed, feeding=False):\\n    # At t=0 the original Feed function is zero: recover reaction terms exactly.\\n    dy = source_model.fedbatch(0, y, par, feed)\\n    F = 0.1 + 0.005*(t-2) if feeding else 0.0\\n    dy[:8] += F * (np.asarray(feed[:8]) - y[:8]) / y[8]\\n    dy[8] = F\\n    return dy\\n\\ndef simulate_yeast(par=None, initial=None, end=50., method='BDF',\\n                   rtol=1e-7, atol=1e-9, feed=None):\\n    \\\"\\\"\\\"Integrate batch, feeding, and full-vessel phases on separate intervals.\\n\\n    Volume capacity is 2 L. The feed polynomial can be integrated analytically;\\n    split at the exact filling time instead of stepping over a state switch.\\n    \\\"\\\"\\\"\\n    from types import SimpleNamespace\\n    par = YeastParameters() if par is None else par\\n    y0 = np.asarray(initial_values() if initial is None else initial, dtype=float)\\n    if y0.shape != (9,) or not np.all(np.isfinite(y0)) or np.any(y0 < 0) or not 0 < y0[8] <= 2:\\n        raise ValueError('Use nine finite nonnegative states and 0 < V0 <= 2 L.')\\n    if not np.isfinite(end) or end <= 0:\\n        raise ValueError('end must be a finite positive time in hours.')\\n    feed = feed_values(y0.tolist()) if feed is None else list(feed)\\n    fill = 2 + (-0.1 + np.sqrt(0.1**2 + 0.01*(2-y0[8])))/0.005\\n    edges = sorted(set([0., float(end)] + [v for v in [2., fill] if 0 < v < end]))\\n    times, states, nfev = [], [], 0\\n    for a, b in zip(edges[:-1], edges[1:]):\\n        feeding = a >= 2 and a < fill and y0[8] < 2-1e-10\\n        grid = np.linspace(a, b, max(3, int((b-a)*12)+1))\\n        sol = solve_ivp(yeast_rhs, (a,b), y0, args=(par,feed,feeding),\\n                        method=method, t_eval=grid, rtol=rtol, atol=atol)\\n        if not sol.success:\\n            raise RuntimeError(sol.message)\\n        start = 1 if times else 0\\n        times.extend(sol.t[start:])\\n        states.extend(sol.y[:, start:].T)\\n        y0 = sol.y[:, -1].copy()\\n        nfev += sol.nfev\\n    return SimpleNamespace(t=np.array(times), y=np.array(states).T,\\n                           success=True, nfev=nfev, fill_time=fill)\\n\\ndef variable_feedstock(seed):\\n    \\\"\\\"\\\"BDG cell 7 ranges, local RNG; preserves zero-width FA draw.\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    bounds = [(35,40),(18,28),(0.6,2),(0,0),(0.5,2),(2,3),(0.2,0.9)]\\n    return [rng.uniform(lo,hi) for lo,hi in bounds] + [5.,0.7]\\n\\ndef variable_growth_parameters(seed):\\n    \\\"\\\"\\\"BDG cell 9: six independent +2 to +5% changes (one-sided).\\\"\\\"\\\"\\n    rng = random.Random(seed)\\n    par = YeastParameters()\\n    for name, baseline in [('numaxG',1.7271),('numaxX',1.6222),\\n                           ('YPSg',0.42),('YPSx',0.24),('YXSg',0.115),('YXSx',0.162)]:\\n        setattr(par, name, baseline*(1+rng.uniform(0.02,0.05)))\\n    return par\\n\\nclass ChemostatParameters:\\n    \\\"\\\"\\\"Values from DAY2/Control_Exercise_2, cell 3; no substituted mu_max.\\\"\\\"\\\"\\n    Ks, Ko, Yxs = 0.1, 0.2e-3, 0.4\\n    A, B, Eg, Gd, R, T = 1e10, 3e90, 58., 550., 8.314e-3, 293.15\\n    kla, Sf, V, F = 250., 10., 2., 0.5\\n    def __init__(self):\\n        self.mux = self.A*np.exp(-self.Eg/(self.R*self.T))/(1+self.B*np.exp(-self.Gd/(self.R*self.T)))\\n        self.O_sat = 0.21*0.027*np.exp(1142/self.T)*32/1000\\n\\ndef chemostat(t, y, par, kla=None):\\n    X, S, O = y\\n    mu = par.mux*S/(par.Ks+S)*O/(par.Ko+O)\\n    D = par.F/par.V\\n    transfer = par.kla if kla is None else kla\\n    return [ (mu-D)*X, D*(par.Sf-S)-mu*X/par.Yxs,\\n             transfer*(par.O_sat-O)-D*O-(1/par.Yxs-1)*mu*X ]\\n\\ndef oxygen_setpoint(t):\\n    return 0.003 if t > 10 else (0.006 if t > 5 else 0.005)\\n\\ndef oxygen_control(end=20., Kc=-50000., tauI=0.02):\\n    \\\"\\\"\\\"Original unsaturated PI; split at setpoint steps; returns actual actuation.\\\"\\\"\\\"\\n    from types import SimpleNamespace\\n    par = ChemostatParameters()\\n    if end <= 0 or tauI <= 0:\\n        raise ValueError('end and tauI must be positive.')\\n    y0 = [3.,0.,0.,0.]\\n    times, states = [], []\\n    edges = sorted(set([0., float(end)] + [v for v in [5.,10.] if v < end]))\\n    for a,b in zip(edges[:-1],edges[1:]):\\n        sp = oxygen_setpoint((a+b)/2)\\n        def rhs(t,y):\\n            X,S,O,I = y\\n            u = par.kla + Kc*(O-sp) + I\\n            return [*chemostat(t,[X,S,O],par,u), Kc/tauI*(O-sp)]\\n        sol = solve_ivp(rhs,(a,b),y0,method='BDF',t_eval=np.linspace(a,b,151),\\n                        rtol=1e-7,atol=[1e-9,1e-9,1e-11,1e-7])\\n        if not sol.success:\\n            raise RuntimeError(sol.message)\\n        start=1 if times else 0\\n        times.extend(sol.t[start:]); states.extend(sol.y[:,start:].T)\\n        y0=sol.y[:,-1]\\n    t=np.array(times); y=np.array(states).T\\n    sp=np.array([oxygen_setpoint(v) for v in t])\\n    u=par.kla + Kc*(y[2]-sp) + y[3]\\n    return SimpleNamespace(t=t,y=y,sp=sp,kla=u,success=True)\\n\",\"course_source/DAY3/model.py\":\"import numpy as np\\r\\n\\r\\ndef initial_values():\\r\\n    ## LIST OF INITIAL CONDITIONS.\\r\\n    # Name     Value     Index     Units\\r\\n    Glu0  =  39.7        # 1       g/L\\r\\n    Xyl0  =  23.5        # 2       g/L\\r\\n    Fur0  =  0.56        # 3       g/L\\r\\n    FA0   =  0           # 4       g/L\\r\\n    HMF0  =  0.2         # 5       g/L\\r\\n    HAc0  =  3.05        # 6       g/L\\r\\n    # Ac0   =  0.0001;    # 7       g/L\\r\\n    EtOH0 =  0.62        # 7       g/L\\r\\n    X0    =  1.75        # 8       g/L\\r\\n    V0    = 0.7          # 9       g/L\\r\\n    \\r\\n    init = [Glu0, Xyl0, Fur0, FA0, HMF0, HAc0, EtOH0, X0, V0]\\r\\n    return init\\r\\n\\r\\ndef feed_values(ini):\\r\\n    feed = ini.copy()\\r\\n    feed[7] = 0             # Xf = 0 !  \\r\\n    return feed\\r\\n\\r\\ndef Feed(t, V):\\r\\n    if t<2 or V>=2:\\r\\n        F = 0\\r\\n    else:\\r\\n        F = 0.1+0.005*(t-2)\\r\\n    return F\\r\\n\\r\\ndef fedbatch(t,x,par, Cfeed):\\r\\n    #  The following 8 components are considered in the model: \\r\\n    #  0. Glucose\\r\\n    #  1. Xylose\\r\\n    #  2. Furfural\\r\\n    #  3. Furfuryl alcohol\\r\\n    #  4. 5-HMF\\r\\n    #  5. HAc\\r\\n    #  6. Ethanol\\r\\n    #  7. Biomass\\r\\n    #  8. Tank volume\\r\\n\\r\\n    #  According to the kinetic model, the following chemical equations describe\\r\\n    #  the system. \\r\\n\\r\\n    #  1. Glucose  -----> Ethanol + Biomass\\r\\n    #  2. Xylose   -----> Ethanol + Biomass\\r\\n    #  3. Furfural -----> Furfuryl alcohol \\r\\n    #  4. HMF      -----> Acetic acid\\r\\n    #  5. HAc     <-----> Ac- [optional - not used here]\\r\\n    #  6. HAc      -----> Biomass (maintenence) \\r\\n\\r\\n\\r\\n    ## 1. STOICHIOMETRIC MATRIX. \\r\\n    # The 8 components and 6 reactions are considered in this model. One could\\r\\n    # also expand the model with Ac-, but that is not implemented.\\r\\n\\r\\n    #                Glu  Xyl  Fur  FA           HMF  HAc           EtOH        X  \\r\\n    st = np.array([[-1,   0,   0,   0,           0,   0,            par.YPSg,   par.YXSg],      # Glucose uptake\\r\\n                   [ 0,  -1,   0,   0,           0,   0,            par.YPSx,   par.YXSx],      # Xylose uptake\\r\\n                   [ 0,   0,  -1,   par.Y_FA_Fur,0,   0,            0,          0       ],      # Furfural uptake\\r\\n                   [ 0,   0,   0,   0,          -1,   par.Y_HAc_HMF,0,          0       ],      # HMF uptake\\r\\n                   [ 0,   0,   0,   0,           0,  -1,            0,          0       ]])     # HAc uptake\\r\\n\\r\\n\\r\\n\\r\\n\\r\\n    ## 2. DEFINE INDIVIDUAL KINETICS AND INHIBITION TERMS. \\r\\n    # Ph stands for phenomena, and it includes all the phenomena considered in\\r\\n    # the model, including reaction, inhibition and equilibria. Each of the\\r\\n    # phenomena is implemented separately below, for clarity\\r\\n    # 1. GLUCOSE UPTAKE RATE.\\r\\n    # numaxG: max consumption rate of glucose           s^-1       [1]\\r\\n    # KSPG:   affinity constant glucose                 g/L        [1]\\r\\n    # KiPG:   inhibition constant glucose               g/L        [1]\\r\\n    Ph1 = par.numaxG * x[0] * x[7] / (par.KSPG + x[0] + (x[0]**2)/par.KiPG)\\r\\n    Ph1 = max(0,Ph1)\\r\\n    # 2. XYLOSE UPTAKE RATE.\\r\\n    # numaxX: max consumption rate of xylose            s^-1       [1]\\r\\n    # KSPX:   affinity constant xylose                  g/L        [1]\\r\\n    # KiPX:   inhibition constant xylose                g/L        [1]\\r\\n    Ph2 = par.numaxX * x[1] * x[7] / (par.KSPX + x[1] + (x[1]**2)/par.KiPX)\\r\\n    Ph2 = max(0,Ph2)\\r\\n\\r\\n    # 3. FURFURAL UPTAKE RATE.\\r\\n    # numaxFur: max uptake rate of furfural               s^-1       [2]\\r\\n    # KSFur:    affinity constant furfural                g/L        [2]\\r\\n    Ph3 = par.numaxFur * x[2] * x[7] / (par.KSFur + x[2])\\r\\n    Ph3 = max(0,Ph3)\\r\\n\\r\\n    # 4. FURFURAL CONVERSION INTO FURFURYL ALCOHOL.\\r\\n    # Y_FA_Fur:  yield coefficient FA/Fur                  gFA/gFur   [2]\\r\\n    Ph4 = Ph3 * par.Y_FA_Fur\\r\\n    Ph4 = max(0,Ph4)\\r\\n\\r\\n    # 5. FURFURYL ALCOHOL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiFAg:    inhibition constant of FA on Glu uptake   g/L        [2]\\r\\n    Ph5 = 1 / (1 + x[3]/par.KiFAg)\\r\\n    Ph5 = max(0,Ph5)\\r\\n\\r\\n    # 6. FURFURYL ALCOHOL INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiFAx:    inhibition constant of FA on Xyl uptake   g/L        [2]\\r\\n    Ph6 = 1 / (1 + x[3]/par.KiFAx)\\r\\n    Ph6 = max(0,Ph6)\\r\\n\\r\\n    # 7. FURFURAL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiFurg:   inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n    Ph7 = 1 / (1 + x[2]/par.KiFurg)\\r\\n    Ph7 = max(0,Ph7)\\r\\n\\r\\n    # 8. FURFURAL INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiFurx:   inhibition constant of Fur on Xyl uptake  g/L        [2]\\r\\n    Ph8 = 1 / (1 + x[2]/par.KiFurx)\\r\\n    Ph8 = max(0,Ph8)\\r\\n\\r\\n    # 9. FURFURAL INHIBITS HMF UPTAKE RATE.\\r\\n    # KiFurHMF: inhibition constant of Fur on HMF         g/L        [2]\\r\\n    Ph9 = 1 / (1 + x[2]/par.KiFurHMF)\\r\\n    Ph9 = max(0,Ph9)\\r\\n\\r\\n    # 10. HMF INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiHMFg:   inhibition constant of HMF on Glu         g/L        [2]\\r\\n    Ph10 = 1 / (1 + x[4]/par.KiHMFg)\\r\\n    Ph10 = max(0,Ph10)\\r\\n\\r\\n    # 11. HMF INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiHMFx:   inhibition constant of HMF on Xyl         g/L        [2]\\r\\n    Ph11 = 1 / (1 + x[4]/par.KiHMFx)\\r\\n    Ph11 = max(0,Ph11)\\r\\n\\r\\n    # 12. HMF UPTAKE RATE. \\r\\n    # numaxHMF: max uptake rate of HMF                    s^-1       [2]\\r\\n    # KSHMF:    affinity constant HMF                     g/L        [2]\\r\\n    Ph12 = par.numaxHMF * x[7] * x[4] / (par.KSHMF + x[4])\\r\\n    Ph12 = max(0,Ph12)\\r\\n\\r\\n    #  13. pH INFLUENCES THE PAIR HAc/Ac.\\r\\n    # pH = 5.5;              % pH is controlled\\r\\n    # pKa = 4.75;            % pKa of acetic acid.\\r\\n    # Ph13 = x(7,1) / x(6,1) * 1 / (1 + 10^(-pH)/10^(-pKa));\\r\\n    # Ph13 = max(0,Ph13);\\r\\n\\r\\n\\r\\n    # 14. HAc UPTAKE RATE.\\r\\n    # numaxHAc: max uptake rate of HAc                    s^-1       [2]\\r\\n    # KSHAc:    affinity constant HAc                     g/L        [2]\\r\\n    Ph14 = x[7] * x[5] * par.numaxHAc / (par.KSHAc + x[5])\\r\\n    Ph14 = max(0,Ph14)\\r\\n\\r\\n\\r\\n    #15. HMF CONVERSION INTO HAc\\r\\n    #Y_HAc_HMF = p(21);     % yield coefficient HAc/HMF                 gHAc/gHMF  [2], [3]\\r\\n    #Ph15 = Y_HAc_HMF * Ph12;\\r\\n    #Ph15 = max(0,Ph15);\\r\\n\\r\\n    # 16. HAc INHIBITS GLUCOSE UPTAKE RATE.\\r\\n    # KiHAcg:   inhibition constant of HAc on Glu         g/L        [2]\\r\\n    Ph16 = 1 / (1 + x[5]/par.KiHAcg)\\r\\n    Ph16 = max(0,Ph16)\\r\\n\\r\\n    # 17. HAc INHIBITS XYLOSE UPTAKE RATE.\\r\\n    # KiHAcx:   inhibition constant of HAc on Xyl         g/L         [2]\\r\\n    Ph17 = 1 / (1 + x[5]/par.KiHAcx)\\r\\n    Ph17 = max(0,Ph17)\\r\\n\\r\\n    #  18. PRODUCTION OF ETHANOL FORM GLUCOSE AND XYLOSE.\\r\\n    # YPSg:     yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n    # YPSx:     yield ethanol-xylose                      gEtOH/gGlu [1]\\r\\n    # Ph18a = par.YPSg * Ph1;\\r\\n    # Ph18b = par.YPSx * Ph2;\\r\\n    # Ph18a = max(0,Ph18a);\\r\\n    # Ph18b = max(0,Ph18b);\\r\\n\\r\\n    # 19. ETHANOL INHIBITS THE UPTAKE OF GLUCOSE AND XYLOSE.\\r\\n    # PMPg:     inhibition constant for glucose           g/L        [1]\\r\\n    # gammaG:   exponent factor inhibition glucose        g/L        [1]\\r\\n    # PMPx:     inhibition constant for xylose            g/L        [1]\\r\\n    # gammaX:   exponent factor inhibition xylose         g/L        [1]\\r\\n    Ph19a = 1-(x[6]/par.PMPg)**par.gammaG\\r\\n    Ph19b = 1-(x[6]/par.PMPx)**par.gammaX\\r\\n    Ph19a = max(0,Ph19a)\\r\\n    Ph19b = max(0,Ph19b)\\r\\n\\r\\n    #  20. CELL GROWTH.\\r\\n    # mGlu:     maintenance constant from glucose         g/L        [1]\\r\\n    # mXyl:     maintenance constant from xylose          g/L        [1]\\r\\n    # YXSg:     yield X-Glu                               gX/gGlu    [1]\\r\\n    # YXSx:     yield X-Xyl                               gX/gXyl    [1]\\r\\n    # mumaxG:   max growth of X from Glu                  h-1        [1]\\r\\n    # mumaxX:   max frowth of X from Xyl                  h-1        [1]\\r\\n\\r\\n    # Ph20a = max(0,(Ph1 + mGlu*x(8,1))*YXSg);\\r\\n    # Ph20b = max(0,(Ph1 + mXyl*x(8,1))*YXSx);\\r\\n\\r\\n    # 21. CATABOLITE REPRESSION.\\r\\n    # KiGlu:    inhibition constant of Glu to Xyl         g/L        Unknown\\r\\n    Ph21 = 1 / (1 + x[0]/par.KiGlu)\\r\\n\\r\\n    # 22. ACETATE IS USED FOR MAINTENANCE.\\r\\n    # mHAc:     maintenance constant from HAc              g/L        Unknown\\r\\n    # YXSHAc:   yield acetate biomass                      g/L        Unknown\\r\\n    Ph22 = max(0,par.mHAc * x[7] * par.YXSHAc)\\r\\n\\r\\n\\r\\n\\r\\n    ## 3. REACTION RATES. \\r\\n    # This includes the reaction rates with inhibition terms.\\r\\n    rates = np.zeros(5)\\r\\n    rates[0]  = Ph1 * Ph5 * Ph7 * Ph10 * Ph16 * Ph19a                            # Glucose uptake rate considering inhibitions.\\r\\n    rates[1]  = Ph2 * Ph6 * Ph8 * Ph11 * Ph17 * Ph19b * Ph21                     # Xylose uptake rate considering inhibitions.\\r\\n    rates[2]  = Ph3                                                              # Furfural uptake rate.\\r\\n    rates[3]  = Ph12 *  Ph9                                                      # 5-HMF uptake rate considering inhibitions.\\r\\n    rates[4]  = Ph14\\r\\n    \\r\\n    ## 4. MASS BALANCE FOR THE SYSTEM.\\r\\n    # Cfeed = feed_values()\\r\\n    dxdt = np.zeros(9)\\r\\n    r, c = st.shape\\r\\n    for i in range(c):\\r\\n        dxdt[i] = sum([rates[j] * st[j,i] for j in range(r)]) + Feed(t, x[8])*(Cfeed[i]-x[i])/x[8]        # Components mass balance -> FedBatch\\r\\n    dxdt[8] =  Feed(t, x[8])                 # Tank volume mass balance\\r\\n    return dxdt\\r\\n\\r\\nif __name__=='__main__':\\r\\n    from parameters import Parameters\\r\\n    init = initial_values()\\r\\n    par = Parameters()\\r\\n    dxdt = fedbatch(0, init, par)\\r\\n    print(dxdt)\\r\\n\\r\\n\\r\\n\\r\\n\",\"course_source/DAY3/parameters.py\":\"# [1] Krishnan et al, 1999\\r\\n# [2] Hanly et al, 2004\\r\\n# [3] Mauricio-Iglesias et al, \\r\\n\\r\\nclass Parameters():\\r\\n    '''Define all Parameters to be used in the model in a class'''\\r\\n    def __init__(self):\\r\\n        ## 1. GLUCOSE UPTAKE RATE.\\r\\n        # numaxG = 2.6808  ;       # max consumption rate of glucose (1.0005)         h^-1       [1]\\r\\n        self.numaxG = 1.7271          # max consumption rate of glucose (1.0005)  h^-1       [1]\\r\\n        # self.KSPG   = 0.14637         # affinity constant glucose to ethanol      g/L        [1]\\r\\n        self.KSPG   = 0.565         # affinity constant glucose to ethanol      g/L        [1]\\r\\n        # self.KiPG   = 4752.8          # inhibition constant glucose to ethanol    g/L        [1]\\r\\n        self.KiPG   = 4890          # inhibition constant glucose to ethanol    g/L        [1]\\r\\n        # p(1:3) = [numaxG, KSPG, KiPG];\\r\\n\\r\\n        ## 2. XYLOSE UPTAKE RATE.\\r\\n        #numaxX = 4.4851;         # max consumption rate of xylose            h^-1       [1]\\r\\n        # self.numaxX = 7.0531          # max consumption rate of xylose            h^-1       [1]\\r\\n        self.numaxX = 1.6222          # max consumption rate of xylose            h^-1       [1]\\r\\n        # self.KSPX   = 0.58763         # affinity constant xylose to ethanol       g/L        [1]\\r\\n        self.KSPX   = 3.4         # affinity constant xylose to ethanol       g/L        [1]\\r\\n        # self.KiPX   = 21.608          # inhibition constant xylose to ethanol     g/L        [1]\\r\\n        self.KiPX   = 18.1          # inhibition constant xylose to ethanol     g/L        [1]\\r\\n        # p(4:6) = [numaxX, KSPX, KiPX];\\r\\n\\r\\n        ## 3. FURFURAL UPTAKE RATE.\\r\\n        # self.numaxFur = 0.14309       # max uptake rate of furfural               h^-1       [2]\\r\\n        self.numaxFur = 4.67e-5*3600       # max uptake rate of furfural               h^-1       [2]\\r\\n        self.KSFur    = 0.05          # affinity constant furfural                g/L        [2]\\r\\n        #p(7:8) = [numaxFur, KSFur];\\r\\n\\r\\n        ## 4. FURFURAL CONVERSION INTO FURFURYL ALCOHOL.\\r\\n        self.Y_FA_Fur = 1.02          # yield coefficient FA/Fur                  gFA/gFur   [2]\\r\\n        #p(9) = Y_FA_Fur;\\r\\n        \\r\\n        ## 5. FURFURYL ALCOHOL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        self.KiFAg = 5                # inhibition constant of FA on Glu uptake   g/L        [2]3\\r\\n        #p(10) = KiFAg;\\r\\n\\r\\n        ## 6. FURFURYL ALCOHOL INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiFAx = 6                # inhibition constant of FA on Xyl uptake   g/L        [2]\\r\\n        #p(11) = KiFAx;\\r\\n\\r\\n        ## 7. FURFURAL INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        # self.KiFurg = 0.4529          # inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n        self.KiFurg = 0.75            # inhibition constant of Fur on Glu uptake  g/L        [2]\\r\\n        #p(12) = KiFurg;\\r\\n\\r\\n        ## 8. FURFURAL INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiFurx = 0.35            # inhibition constant of Fur on Xyl uptake  g/L        [2]\\r\\n        #p(13) = KiFurx;\\r\\n\\r\\n        ## 9. FURFURAL INHIBITS HMF UPTAKE RATE.\\r\\n        self.KiFurHMF = 0.25          # inhibition constant of Fur on HMF         g/L        [2]\\r\\n        #p(14) = KiFurHMF;\\r\\n\\r\\n        ## 10. HMF INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        self.KiHMFg = 2               # inhibition constant of HMF on Glu         g/L        [2]\\r\\n        #p(15) = KiHMFg;\\r\\n\\r\\n        ## 11. HMF INHIBITS XYLOSE UPTAKE RATE.\\r\\n        self.KiHMFx = 10              # inhibition constant of HMF on Xyl         g/L        [2]\\r\\n        #p(16) = KiHMFx;\\r\\n\\r\\n        ## 12. HMF UPTAKE RATE. \\r\\n        # self.numaxHMF = 0.3154        # max uptake rate of HMF                    h^-1       [2]\\r\\n        self.numaxHMF = 8.76E-5*3600        # max uptake rate of HMF                    h^-1       [2]\\r\\n        self.KSHMF = 0.5              # affinity constant HMF                     g/L        [2]\\r\\n        #p(17:18) = [numaxHMF, KSHMF];\\r\\n\\r\\n        ## 13. pH INFLUENCES THE PAIR HAc/Ac.\\r\\n        self.pH = 5.5                 # pH is controlled\\r\\n        self.pKa = 4.75               # pKa of acetic acid.\\r\\n\\r\\n        ## 14. HAc UPTAKE RATE.\\r\\n        self.numaxHAc = 1.23E-5*3600*0.001     # max uptake rate of HAc                    h^-1       [2]\\r\\n        self.KSHAc = 2.5                 # affinity constant HAc                     g/L        [2]\\r\\n        #p(19:20) = [numaxHAc, KSHAc];\\r\\n\\r\\n        ## 15. HMF CONVERSION INTO HAc\\r\\n        self.Y_HAc_HMF = 0.534        # yield coefficient HAc/HMF                 gHAc/gHMF  [2], [3]\\r\\n        #p(21) = Y_HAc_HMF;\\r\\n\\r\\n        ## 16. HAc INHIBITS GLUCOSE UPTAKE RATE.\\r\\n        # KiHAcg = 5.1827;         # inhibition constant of HAc on Glu         g/L        [2]\\r\\n        self.KiHAcg = 5.6703          # inhibition constant of HAc on Glu         g/L        [2]\\r\\n        # p(22) = KiHAcg;\\r\\n\\r\\n        ## 17. HAc INHIBITS XYLOSE UPTAKE RATE.\\r\\n        # KiHAcx = 0.73;           # inhibition constant of HAc on Xyl         g/L        [2]\\r\\n        self.KiHAcx = 3.8291          # inhibition constant of HAc on Xyl         g/L        [2]\\r\\n        # p(23) = KiHAcx;\\r\\n\\r\\n        ## 18. PRODUCTION OF ETHANOL FORM GLUCOSE AND XYLOSE.\\r\\n        # YPSg = 0.42;             # yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n        self.YPSg = 0.42              # yield ethanol-glucose                     gEtOH/gGlu [1]\\r\\n\\r\\n        self.YPSx = 0.24              # yield ethanol-xylose                      gEtOH/gGlu [1]\\r\\n        # p(24:25) = [YPSg, YPSx];\\r\\n\\r\\n        ## 19. ETHANOL INHIBITS THE UPTAKE OF GLUCOSE AND XYLOSE.\\r\\n        self.PMPg = 103             # inhibition constant for glucose           g/L        [1]\\r\\n        self.gammaG = 1.42            # exponent factor inhibition glucose        g/L        [1]\\r\\n        self.PMPx = 60.2           # inhibition constant for xylose            g/L        [1]\\r\\n        # PMPx = 20.9778;          # inhibition constant for xylose            g/L        [1]\\r\\n        self.gammaX =0.608          # exponent factor inhibition xylose         g/L        [1]\\r\\n        # gammaX = 1.7376;         # exponent factor inhibition xylose         g/L        [1]\\r\\n        # p(26:29) = [PMPg, gammaG, PMPx, gammaX];\\r\\n\\r\\n        ## 20. CELL GROWTH.\\r\\n        self.mGlu = 2.69E-5           # maintenance constant from glucose         g/L        [1]\\r\\n        self.mXyl = 1.86E-5           # maintenance constant from xylose          g/L        [1]\\r\\n        self.YXSg = 0.115            # yield X-Glu                               gX/gGlu    [1]\\r\\n        self.YXSx = 0.162             # yield X-Xyl                               gX/gXyl    [1]\\r\\n        self.mumaxG = 0.3308          # max growth of X from Glu                  h-1        [1]\\r\\n        self.mumaxX = 1.0008          # max frowth of X from Xyl                  h-1        [1]\\r\\n        # KSPGgr = 0.565;         # affinity constant glucose to growth       g/L        [1]\\r\\n        # KiPGgr = 283.7;         # inhibition constant glucose to growth     g/L        [1]\\r\\n        # KSPXgr = 3.4;           # affinity constant xylose to growth        g/L        [1]\\r\\n        # KiPXgr = 18.1;          # inhibition constant xylose to growth      g/L        [1]\\r\\n        # PMXg   = 95.4 ;         # max EtOH concentration for growth glucose g/L        [1]\\r\\n        # PMXx   = 59.04;         # max EtOH concentration for growth xylose  g/L        [1]\\r\\n        # p(30:35) = [mGlu, mXyl,YXSg, YXSx, mumaxG, mumaxX];\\r\\n\\r\\n        ## 21. CATABOLITE REPRESSION.\\r\\n        self.KiGlu = 13.763                # inhibition constant of Glu to Xyl         g/L        Unknown\\r\\n        # p(36) = KiGlu;\\r\\n\\r\\n        ## 22. ACETIC ACID IS USED FOR MAINTENANCE.\\r\\n        self.mHAc = 0                 # maintenance constant from HAc              g/L        Unknown\\r\\n        self.YXSHAc = 0               # yield acetate biomass                      g/L        Unknown \\r\\n        # p(38:39) = [mHAc, YXSHAc];\\r\\n\\r\\n\\r\\n        ## COLLECT PARAMETER NAMES\\r\\n        self.par_names = {'numaxG', 'KSPG', 'KiPG', 'numaxX', 'KSPX', 'KiPX', 'numaxFur', 'KSFur', 'Y_FA_Fur', 'KiFAg', 'KiFAx', 'KiFurg',\\r\\n            'KiFurx', 'KiFurHMF', 'KiHMFg', 'KiHMFx', 'numaxHMF', 'KSHMF', 'numaxHAc', 'KSHAc', 'Y_HAC_HMF', 'KiHAcg', 'KiHAcx', \\r\\n            'YPSg', 'YPSx', 'PMPg', 'gammaG', 'PMPx', 'gammaX', 'mGlu', 'mXyl', 'YXSg', 'YXSx', 'KiGlu'}\\r\\n\\r\\n        ## COLLECT DEPENDENT VARIABLE NAMES\\r\\n        self.var_names = ['Glucose', 'Xylose', 'Furfural','Furfuryl alcohol','5-HMF', 'Acetic acid', 'Ethanol', 'Biomass']\\r\\n\\r\\nif __name__=='__main__':\\r\\n    par = Parameters()\\r\\n    print(par.mHAc)\"}")
for _name, _content in _support.items():
    _path = Path(_name)
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print('Model support ready. Requires numpy, scipy, matplotlib.')


## 01. Python as an engineering calculator

**Objective:** Use variables, units, functions, and NumPy arrays to express a calculation you can inspect and reuse.

<p>Start with the original hypotenuse calculation: <code>a = 4.32</code>, <code>b = 3.51</code>, and <code>c = np.sqrt(a**2 + b**2)</code>. Python uses <code>**</code> for powers. Assigning <code>a = 4.32</code> stores a number; it does not attach a unit.</p>
 <div class="equation">c = √(a² + b²)</div>
 <p>If both inputs are metres, the result is metres. A function isolates a calculation from its inputs. <code>return</code> sends a value back; <code>print</code> only displays it. Indentation defines the function body. A NumPy array applies arithmetic element by element.</p>
 <p>The notebook also sums powers using <code>range(1, n+1)</code>. The final bound is excluded, and Python indexing starts at zero. We will use this same indexing convention for concentration states throughout the course.</p>

**Task:** Complete hypotenuse(a, b), evaluate the original inputs, and calculate lengths for three pairs using arrays. Keep the function general: the tests also pass different inputs.

Source: DAY1/Introduction.ipynb, original zero-based cells 1, 3, 5, 7.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def hypotenuse(a, b):
    # TODO: replace this placeholder with the equation above.
    return a

c = hypotenuse(4.32, 3.51)
lengths = hypotenuse(np.array([3., 5., 8.]), np.array([4., 12., 15.]))
print("Original length:", c)
print("Array of lengths:", lengths)


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert np.isclose(c, np.sqrt(4.32**2+3.51**2)), 'Check the square root and powers.'
assert np.isclose(hypotenuse(6,8),10), 'The function must work for other inputs.'
assert np.allclose(lengths,[5,13,17]), 'Use elementwise array operations.'
print("All numerical checks passed.")


**Interpretation:** The original result is approximately 5.5662. Distinguish a Python list from a NumPy array: multiplying a list repeats it, while multiplying an array scales its entries.

**Hand in:** Save your function and state why its inputs must have the same physical units.


## 02. Your first initial-value problem

**Objective:** Translate dy/dt = y into a derivative function and test the numerical solution against exp(t).

<p>An initial-value problem specifies a derivative and a starting value. Here time is dimensionless and <code>y(0) = 1</code>. The derivative function receives <code>(t, y)</code> and returns <em>dy/dt</em>, not the next value of y.</p><div class="equation">dy/dt = y &nbsp; · &nbsp; y(0) = 1 &nbsp; → &nbsp; y(t) = eᵗ</div><p><code>solve_ivp</code> expects the initial state as a sequence, even for one equation. <code>sol.t</code> is a one-dimensional time array; <code>sol.y</code> has shape <code>(n_states, n_times)</code>. Therefore <code>sol.y[0]</code> is the entire trajectory of the first state.</p><p><code>t_eval</code> requests output times. It does not fix the solver's internal step size. Always inspect <code>sol.success</code> before using the result. The original notebook's SymPy solution is an optional symbolic cross-check; the known analytical solution is sufficient here.</p>

**Task:** Replace the zero derivative with y, integrate from 0 to 1 at 51 output times, and compute the maximum absolute error against exp(t).

Source: DAY1/Introduction.ipynb, original zero-based cells 8, 9, 10, 11, 12, 13, 14.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def rhs(t, y):
    return [0.]  # TODO: dy/dt = y

t = np.linspace(0, 1, 51)
sol = solve_ivp(rhs, (0,1), [1.], t_eval=t, rtol=1e-8, atol=1e-10)
exact = np.exp(sol.t)
error = np.max(np.abs(sol.y[0]-exact))
print("Success:", sol.success, "Maximum error:", error)
plt.plot(sol.t, sol.y[0], label="Numerical")
plt.plot(sol.t, exact, "--", label="Analytical")
plt.xlabel("Dimensionless time"); plt.ylabel("y"); plt.legend()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert sol.success, sol.message
assert sol.y.shape == (1,51), 'Check the output shape.'
assert np.allclose(rhs(0,[2.]),[2.]), 'Return the derivative evaluated at the supplied state.'
assert np.isclose(sol.y[0,-1],np.e,rtol=1e-6), 'At t=1, y must be e.'
assert error < 1e-6, 'Compare numerical and analytical solutions.'
print("All numerical checks passed.")


**Interpretation:** A visually matching plot is insufficient evidence of accuracy. Report the error and tolerances. This analytical comparison is your first regression test for an ODE calculation.

**Hand in:** A labelled numerical/analytical plot, solver status, and maximum error.


## 03. Parameters, tolerances & solver choice

**Objective:** Pass parameters correctly and distinguish output spacing from numerical accuracy.

<p>The second example adds a constant A to the derivative. With <code>A = -0.7</code> and <code>y(0) = 1</code>, an integrating factor gives the exact solution below.</p><div class="equation">dy/dt = y + A &nbsp; → &nbsp; y(t) = (y₀ + A)eᵗ − A</div><p>For a function <code>rhs(t, y, A)</code>, pass <code>args=(A,)</code>. The comma makes a one-item tuple. A keyword such as <code>par=par</code> does not pass a model parameter to <code>solve_ivp</code>.</p><p>The error scale for each state is approximately <code>atol + rtol * abs(y)</code>. RK45 is a useful first choice for nonstiff problems; BDF uses an implicit multistep method suited to stiff dynamics. This single equation does not need BDF, but reproducing both methods makes their interface clear.</p>

**Task:** Implement the parameterized derivative. Compare RK45 and BDF at rtol = 1e-7, and record terminal errors and nfev. Then repeat with rtol = 1e-4 and 1e-9 in your own experiment.

Source: DAY1/Introduction.ipynb, original zero-based cells 15, 16.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

A = -0.7
def rhs(t, y, A):
    return [y[0]]  # TODO: include A

errors = {}
for method in ["RK45", "BDF"]:
    sol = solve_ivp(rhs, (0,1), [1.], args=(A,), method=method,
                    rtol=1e-7, atol=1e-10)
    exact_end = (1+A)*np.exp(1)-A
    errors[method] = abs(sol.y[0,-1]-exact_end)
    print(method, "error =", errors[method], "nfev =", sol.nfev)


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert np.allclose(rhs(0,[1.],-0.7),[0.3]), 'Include the parameter in the derivative.'
assert set(errors)=={'RK45','BDF'}
assert max(errors.values()) < 2e-5, 'Check args and the analytical solution.'
print("All numerical checks passed.")


**Interpretation:** The terminal value is approximately 1.51548. Neither method should give 2.71828: that would solve the previous equation and ignore A.

**Hand in:** A table of method, tolerance, terminal error, and function evaluations. Explain why more output points do not necessarily reduce integration error.


## 04. Coupled balances: reversible isomerization

**Objective:** Construct a coupled species balance and verify mass conservation and equilibrium.

<p>Consider a closed, constant-volume system with the reversible reaction A ⇌ B. Both species have the same molar mass, and the model has no inlet, outlet, or side reaction. Use concentrations in mol/L and time in hours.</p><div class="equation">dA/dt = −k₁A + k₂B<br>dB/dt = +k₁A − k₂B</div><p>The two derivatives sum to zero, so A + B stays at its initial value, 4 mol/L. At equilibrium <code>k1*A = k2*B</code>, giving <code>Aeq = 8/3</code> and <code>Beq = 4/3</code> for the original rates <code>k1=1</code>, <code>k2=2</code> h⁻¹ and initial states <code>[3,1]</code>.</p><p><strong>Source correction:</strong> the notebook uses <code>par=par</code> in the solver call and captures an old parameter list in a default argument. Here the function receives explicit rates through <code>args=(k1,k2)</code>.</p>

**Task:** Complete the missing B balance. Solve for 10 h, calculate the maximum conservation residual, and verify the equilibrium composition. Plot both concentrations.

Source: DAY1/Introduction.ipynb, original zero-based cells 17, 18.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def reaction(t, y, k1, k2):
    A, B = y
    return [-k1*A+k2*B, 0.]  # TODO: balance B

k1, k2 = 1., 2.
sol = solve_ivp(reaction, (0,10), [3.,1.], args=(k1,k2),
                t_eval=np.linspace(0,10,201), rtol=1e-8, atol=1e-10)
residual = np.max(np.abs(sol.y.sum(axis=0)-4))
print("Conservation residual [mol/L]:", residual)
print("Final A, B [mol/L]:", sol.y[:,-1])
plt.plot(sol.t, sol.y.T, label=["A", "B"])
plt.xlabel("Time [h]"); plt.ylabel("Concentration [mol/L]"); plt.legend()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert sol.success, sol.message
assert np.allclose(reaction(0,[3,1],1,2),[-1,1]), 'Check the forward and reverse signs.'
assert residual < 1e-7, 'A+B must be constant.'
assert np.allclose(sol.y[:,-1],[8/3,4/3],atol=1e-6), 'Check the equilibrium ratio.'
print("All numerical checks passed.")


**Interpretation:** A falls from 3 toward 2.6667 mol/L; B rises toward 1.3333 mol/L. Conservation tests detect missing signs and terms even when a plot looks plausible.

**Hand in:** Your two-state model, concentration plot, conservation residual, and an independently derived equilibrium.


## 05. What a numerical integrator actually does

**Objective:** Implement forward Euler and measure how the time step changes the integration error.

<p>The harmonic-signal example integrates a sine wave using the previous slope. With a unit amplitude, zero bias, and initial integral zero, the exact accumulated signal is <code>1 - cos(t)</code>.</p><div class="equation">dz/dt = sin(t), z(0) = 0<br>zₙ₊₁ = zₙ + Δt sin(tₙ)</div><p>Forward Euler uses one slope per step. Its global error is generally proportional to Δt, so halving the step should approximately halve the error in this smooth example. Evaluating the signal at the new time instead would implement a different rule.</p><p>Allocate arrays before a loop and start updating at index 1; the first entry is the initial condition. The notebook's plotting and subplot examples are used here to compare numerical trajectories and their errors on aligned axes.</p>

**Task:** Complete the Euler update. Integrate to 10 with dt = 0.2, 0.1, 0.05, and 0.025. Compare maximum errors and plot the finest trajectory against the exact integral.

Source: DAY1/Introduction.ipynb, original zero-based cells 20, 22, 23, 24.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def euler(dt):
    t = np.linspace(0,10,round(10/dt)+1)
    z = np.zeros(len(t))
    for i in range(1,len(t)):
        z[i] = z[i-1]  # TODO: add previous slope times dt
    return t,z

steps = [0.2,0.1,0.05,0.025]
errors = []
for dt in steps:
    t,z = euler(dt)
    errors.append(np.max(np.abs(z-(1-np.cos(t)))))
print("dt / max error:", list(zip(steps,errors)))
plt.plot(t,z,label="Euler"); plt.plot(t,1-np.cos(t),"--",label="Exact")
plt.xlabel("Dimensionless time"); plt.ylabel("Integrated signal"); plt.legend()


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert len(errors)==4
assert errors[-1] < 0.02, 'Use the previous derivative in the Euler update.'
assert 1.7 < errors[0]/errors[1] < 2.3, 'The global error should approximately halve.'
assert np.all(np.diff(errors)<0), 'Refining the step should improve this example.'
print("All numerical checks passed.")


**Interpretation:** A fine output grid and a fine integration step are different concepts. Euler directly uses dt; solve_ivp chooses its own steps. This distinction matters when fast oxygen transfer enters the model on Day 2.

**Hand in:** An error-versus-step table and a one-sentence explanation of first-order convergence.


## 06. Day 1 lab: verify a dynamic model

**Objective:** Apply the solve–inspect–verify workflow independently, and recognize when an explicit scheme becomes unstable.

<p>Use the reversible reaction as your assessed model. Its transient decays with rate <code>k1+k2</code>. Subtracting the equilibrium gives <code>d(A-Aeq)/dt = -(k1+k2)(A-Aeq)</code>.</p><div class="equation">A(t) = Aeq + (A₀ − Aeq)e<sup>−(k₁+k₂)t</sup><br>Euler stability: |1 − (k₁+k₂)Δt| &lt; 1</div><p>For the original rates, Euler needs Δt &lt; 2/3 h for linear stability. This is a stability condition, not an accuracy guarantee. A stable simulation can still have substantial error.</p><p>The notebook ends with a class-based Van der Pol oscillator. That is an optional extension: <code>dx1/dt=x2</code>, <code>dx2/dt=mu*(1-x1**2)*x2-x1</code>, with <code>mu=10</code> and initial <code>[2,0]</code>. Re-implement it with <code>solve_ivp(method='BDF')</code> after completing the core lab.</p>

**Task:** Write the reaction derivative without copying the previous cell. Compute Aeq and the exact trajectory. Compare RK45 and BDF, and verify both errors and total concentration.

Source: DAY1/Introduction.ipynb, original zero-based cells 18, 25, 26.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

k1, k2 = 1., 2.
def rhs(t,y):
    # TODO: return both species balances
    return [0.,0.]

t = np.linspace(0,10,201)
Aeq = 8/3
exact = Aeq+(3-Aeq)*np.exp(-(k1+k2)*t)
results = {}
for method in ["RK45","BDF"]:
    sol = solve_ivp(rhs,(0,10),[3.,1.],method=method,t_eval=t,rtol=1e-8,atol=1e-10)
    results[method] = np.max(np.abs(sol.y[0]-exact))
    print(method, "A error:", results[method], "total residual:", np.max(abs(sol.y.sum(axis=0)-4)))


### Check your result
Run this after completing the exercise. An AssertionError indicates a result to investigate.


In [ ]:
assert np.allclose(rhs(0,[3.,1.]),[-1,1]), 'Implement both balances.'
assert max(results.values()) < 2e-6, 'Your model must match the transient, not just equilibrium.'
assert np.max(abs(sol.y.sum(axis=0)-4)) < 1e-7
print("All numerical checks passed.")


**Interpretation:** Verification requires three independent facts: the initial condition, the transient solution, and the conserved total. Passing only an equilibrium check can hide an incorrect rate constant.

**Hand in:** Submit a short notebook containing the balances, units, exact solution, solver comparison, and an Euler stability argument. Extension: compare RK45/BDF on the oscillator.
